# 09. M3 Max: последовательный distill 5s

Экспериментальный локальный маршрут: Qwen → сохранённый conditioning → DiT MPS → сохранённые latents → Hunyuan CPU decoder → MP4. Реальный запуск включается вручную; полная генерация может быть долгой.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladislav-vasilenko/generative-ai-research/blob/main/kandinsky-5/notebooks/09_local_m3_max.ipynb)

### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Сначала проверка MPS

На M3 Max 48 ГБ память общая. Не отключаем системные memory limits. BF16 нужен для разумной памяти Qwen/DiT; проверим matmul, SDPA и autocast. Conv3d декодер оставляем на CPU FP32. Успех маленького smoke test не доказывает, что весь 5s запуск поместится.


In [ ]:
MPS_AVAILABLE=torch.backends.mps.is_available()
print('MPS available:',MPS_AVAILABLE)
if MPS_AVAILABLE:
    import torch.nn.functional as F
    a=torch.randn(1,2,8,16,device='mps',dtype=torch.bfloat16)
    with torch.autocast('mps',dtype=torch.bfloat16): out=F.scaled_dot_product_attention(a,a,a)
    torch.mps.synchronize(); print(out.shape,out.dtype,torch.isfinite(out).all().item())
    del a,out; release()


### 2. Forward настоящих классов на Metal

Эта проверка использует tiny DiT: она ловит неподдерживаемые операции, а не проверяет pretrained weights. Если ошибка возникает здесь, сначала разберите traceback; переход к большим весам её не исправит.


In [ ]:
if MPS_AVAILABLE:
    from labkit.native import tiny_dit
    m=tiny_dit('mps')
    with torch.inference_mode():
        y=m(torch.randn(1,4,4,33,device='mps'),torch.randn(1,6,32,device='mps'),torch.randn(1,16,device='mps'),torch.tensor([500.],device='mps'),[torch.arange(1,device='mps'),torch.arange(2,device='mps'),torch.arange(2,device='mps')],torch.arange(6,device='mps'))
    print('tiny MPS velocity:',y.shape,'finite:',torch.isfinite(y).all().item())
    del m,y; release()


### 3. Скачивание только выбранных компонентов

Это десятки ГБ на диске. Перечень и pinned revisions сохранены в reference/hf-revisions.json. Токен Hugging Face берётся из стандартного окружения HF_TOKEN/login, в notebook его не печатаем. Qwen/CLIP/VAE скачиваются отдельно от DiT.


In [ ]:
DOWNLOAD_WEIGHTS=False
if DOWNLOAD_WEIGHTS:
    from labkit.components import download_component
    for name in ['qwen','clip','dit','vae']: print(name,download_component(name))
else: print('Веса не скачивались. Для загрузки задайте DOWNLOAD_WEIGHTS=True.')


### 4. Проверим файлы до запуска

strict state_dict loading защищает от случайного смешивания SFT/distill архитектуры. Путь должен содержать именно distilled16steps_5s. Загрузка внутри model с meta parameter placeholders уменьшает лишнюю временную копию весов.


In [ ]:
required=[ROOT/'weights/dit/model/kandinsky5lite_t2v_distilled16steps_5s.safetensors',ROOT/'weights/qwen/config.json',ROOT/'weights/clip/config.json',ROOT/'weights/vae/vae/config.json']
for p in required: print(p.relative_to(ROOT),p.exists())


### 5. Первый небольшой запуск

1s 256×256, 16 steps — проверка формы/ресурсов, вне официального списка resolutions. Качество такого вывода не характеризует distill checkpoint. Малый запуск не заменяет финальный 5s 512×768. Expansion выключен; guidance=1; scale=5 задан явно.


In [ ]:
RUN_SMALL=False
if RUN_SMALL:
    assert MPS_AVAILABLE,'Нужен MPS'
    assert all(p.exists() for p in required),'Сначала скачайте компоненты'
    from labkit.components import sequential_t2v
    path=sequential_t2v('A red toy car slowly moves across a wooden table.',height=256,width=256,seconds=1,device='mps',output=ROOT/'artifacts/mac_small.mp4')
    from IPython.display import Video,display
    display(Video(str(path),embed=True))
else: print('Малый pretrained запуск пропущен.')


### 6. Финальный 5s маршрут

Запускаем только после малого теста. Каждый компонент удаляется перед следующим; сохранённые conditioning и latents позволяют не повторять предыдущую стадию при сбое декодирования. JSON рядом с MP4 содержит версии, seed, NFE и stage timings. CPU VAE decoding может стать самым медленным этапом.


In [ ]:
RUN_FULL_5S=False
if RUN_FULL_5S:
    assert MPS_AVAILABLE and all(p.exists() for p in required)
    from labkit.components import sequential_t2v
    path=sequential_t2v('A cat in a red hat slowly turns its head. Static camera, warm daylight.',output=ROOT/'artifacts/mac_distill_5s.mp4')
    from IPython.display import Video,display
    display(Video(str(path),embed=True))
else: print('Полный 5s инференс НЕ выполнялся.')


### 7. Возобновление только decoder

Если latents уже сохранены, повторный DiT не нужен. torch.load(weights_only=True) ограничивает содержимое загружаемого файла. Загружайте собственные tensor artifacts. При изменении VAE версии старые latents требуют проверки совместимости.


In [ ]:
RUN_RESUME_DECODE=False
if RUN_RESUME_DECODE:
    from labkit.components import decode_latents
    latent=torch.load(ROOT/'artifacts/mac_distill_5s.latents.pt',map_location='cpu',weights_only=True)
    frames=decode_latents(latent)
    import imageio.v2 as imageio
    imageio.mimwrite(ROOT/'artifacts/mac_resumed.mp4',frames,fps=24,codec='libx264')
else: print('Resume decode пропущен.')


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
